# 립리딩 전처리 · 학습 파이프라인 (Colab)

**실험용이다.** 배포 모델은 `colab_release.ipynb`에서 굽는다.

위에서부터 실행하면 된다. 전처리(2절)는 크롭 규격을 바꿀 때만 돌리고,
평소에는 1절 셋업 다음 3절로 바로 간다.

| 절 | 내용 |
|---|---|
| 1 | 셋업 — 마운트·저장소·규격검증·로컬복사·매니페스트 |
| 2 | 전처리 — 규격을 바꿀 때만 |
| 3 | 학습 — 실험 러너 |
| 4~ | 실험 (증강·랜드마크·화자 제외·오디오 증류) |
| 부록 | 오답 분석 (손으로 실행) |

## 1. 셋업

드라이브 마운트 · 저장소 받기 · npy 규격 검증 · 로컬 복사 · 매니페스트 생성을
한 번에 한다. 끊겨도 다시 돌리면 이어받는다.

**런타임을 새로 켜면 이 셀만 돌리면 된다.**

In [73]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_RAW = DRIVE_ROOT / "raw"
# 크롭 규격마다 폴더가 다르다. 섞으면 백본이 첫 배치에서 죽는다.
#   processed_f60     112x80 · 축 정렬 상자        (옛 규격)
#   processed_align   192x96 · 닮음 변환 정렬      (2026-08-24~, 현재)
PROCESSED = DRIVE_ROOT / "processed_align"
PROCESSED_F60 = PROCESSED   # 아래 절들이 쓰는 옛 이름 호환

n_drive = len(list(PROCESSED.glob("*.npy")))
print("Drive", PROCESSED.name + ":", n_drive, "개")
assert n_drive == 1238, f"{PROCESSED.name}이 1238개가 아니다"

os.chdir("/content")
for junk in Path("/content").glob("*REPO_DIR*"):
    shutil.rmtree(junk, ignore_errors=True)
    print("잘못 만들어진 폴더 삭제:", junk.name)

REPO = "/content/hanium-lipreading"
REPO_DIR = Path(REPO)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "develop"], check=True)
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-b", "develop",
                    "https://github.com/HumanRhoid/hanium-lipreading.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "wandb"], check=True)

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED, manifest_f60=manifest_f60)

import numpy as np
from src.ml.models.backbone import LipReadingBackbone
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT

# (프레임, 높이, 너비, 채널). 192x96 이면 (60, 96, 192, 3)이다.
WANT = (FIXED_FRAME_COUNT, LipReadingBackbone.input_height,
        LipReadingBackbone.input_width, 3)


def npy_shape(folder):
    """폴더의 npy 한 장을 열어 규격을 본다. 비어 있으면 None."""
    files = sorted(folder.glob("*.npy")) if folder.exists() else []
    return np.load(files[0]).shape if files else None

drive_shape = npy_shape(PROCESSED)
assert drive_shape == WANT, f"드라이브 규격 불일치: {PROCESSED.name} {drive_shape} · 기대 {WANT}"

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()

# 개수만 보면 옛 규격이 같은 개수로 남아 있을 때 복사를 건너뛴다. 실제로 그
# 사고가 났다(로컬에 112x80 1238개가 남아 재복사가 안 됨). 규격까지 본다.
local_shape = npy_shape(LOCAL_F60)
if local_shape is not None and local_shape != WANT:
    print("옛 규격 로컬 사본 제거:", local_shape)
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)

# Drive FUSE는 파일 1238개를 한 번에 긁으면 자주 끊긴다(Errno 107 Transport
# endpoint is not connected). copytree는 한 장만 실패해도 통째로 무너지므로
# 직접 채우고 실패분을 재시도한다. 끊겨도 다시 돌리면 이어받는다.
#
# 있는지만 보면 안 된다. 끊길 때 만들어지다 만 0바이트 파일이 남는데 그것도
# exists()는 참이라 건너뛴다. 모든 npy가 같은 규격이라 크기도 같으므로
# 원본 한 장의 크기와 대조해 덜 받은 것을 가려낸다.
LOCAL_F60.mkdir(parents=True, exist_ok=True)
sources = sorted(PROCESSED.glob("*.npy"))
REF_SIZE = sources[0].stat().st_size

def needs_copy(src):
    dst = LOCAL_F60 / src.name
    return not dst.exists() or dst.stat().st_size != REF_SIZE

def fetch(src):
    """성공하면 None, 실패하면 그 원본을 돌려준다."""
    try:
        shutil.copy2(src, LOCAL_F60 / src.name)
        return None
    except OSError:
        return src

# 드라이브는 한 장씩 받으면 지연이 대부분이라 동시에 받는 편이 몇 배 빠르다.
todo = [f for f in sources if needs_copy(f)]
for attempt in range(1, 4):
    if not todo:
        break
    print(f"복사 {len(todo)}개 · {len(todo) * REF_SIZE / 1e9:.1f} GB (시도 {attempt})")
    t0, done, failed = time.time(), 0, []
    with ThreadPoolExecutor(16) as pool:
        for result in pool.map(fetch, todo):
            done += 1
            if result is not None:
                failed.append(result)
            if done % 200 == 0:
                sec = max(time.time() - t0, 0.001)
                print(f"  {done}/{len(todo)} · {sec:.0f}초 · "
                      f"{done * REF_SIZE / 1e6 / sec:.0f} MB/s")
    todo = failed
    if todo:
        print("  실패", len(todo), "개 - 재시도")
if todo:
    raise RuntimeError(f"복사 실패 {len(todo)}개. 드라이브 마운트가 끊긴 것이니 "
                       "런타임을 다시 시작하고 이 셀을 다시 돌릴 것. 받은 것은 남는다.")

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size != REF_SIZE]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, f"로컬 복사 불완전 · 크기 이상 {len(bad)}개"
assert npy_shape(LOCAL_F60) == WANT, "복사 후에도 규격이 안 맞는다"
print("규격 확인", WANT)
# end

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1238 개
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1238개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1238 개 · 0 초 · 0바이트 0 개


## 2. 전처리 — 영상을 .npy로 (규격을 바꿀 때만)

`processed_align/`이 이미 차 있으면 건너뛴다. 1절이 개수와 규격을 검사하므로
여기서 다시 구울 이유는 크롭 방식을 바꿨을 때뿐이다.

**규격을 바꿨으면 반드시 새 폴더에 굽는다.** `run_batch`는 이미 있는 파일을
건너뛰므로 같은 폴더에 다시 돌리면 옛 규격이 그대로 남는다.

In [ ]:
# 전처리에만 필요하다. 이미 구운 npy를 쓸 거면 이 절은 건너뛴다.
!pip install --quiet mediapipe opencv-python

LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)
if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}
print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 크롭 규격을 바꿨으면 반드시 새 폴더에 굽는다. run_batch는 이미 있는 파일을
# 건너뛰므로 같은 폴더에 다시 돌리면 옛 규격이 그대로 남고, 개수도 그대로라
# 다음 셀의 검사도 통과해 버린다.
PROCESSED_ALIGN = DRIVE_ROOT / "processed_align"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT, TARGET_HEIGHT, TARGET_WIDTH

print(f"규격 {TARGET_WIDTH}x{TARGET_HEIGHT} · {FIXED_FRAME_COUNT}프레임 -> {PROCESSED_ALIGN}")
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_ALIGN,
                  frames=FIXED_FRAME_COUNT)

## 3. 학습 — 실험 러너

`run_experiment.py`는 별도 프로세스로 돌고 설정을 결과에 함께 기록한다.
커널에 남은 몽키패치에 오염되지 않고, 기준값을 손으로 적을 일이 없다.

- 결과는 `--results-dir`에 `<이름>.json`으로 쌓인다. **드라이브를 가리켜야
  세션이 끝나도 남는다.** 같은 이름으로 다시 돌리면 끝난 칸은 건너뛴다
- 이전 런과 설정이 다르면 **학습을 걸기 전에** 멈춘다
- `--summary`는 학습 없이 요약만, `--baseline <이름>`으로 기준선과 견준다
- 비교는 마지막 에폭 값으로 한다. 양쪽에 없으면 저장값으로 물러서며 그 사실을 찍는다
- cuDNN 결정성은 **기본 켬**이다. 2026-08-25 재측정에서 조건을 안 바꿨는데도
  s01 평균 -0.030이 어긋났다

**이름을 그대로 두고 다시 돌리면 이미 잰 것을 또 재게 된다.** align 24런(약 9시간)이
cv192와 같은 조건이었던 사고가 그렇게 났다.

`train()`은 딕셔너리를 반환한다. 발표 수치는 `["best"]`가 아니라 `["last"]`다.

In [ ]:
# 러너가 쓸 기준선을 드라이브로 옮긴다. 저장소 results/ 는 세션과 함께 사라진다.
import shutil
from pathlib import Path

RESULTS = DRIVE_ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
for f in sorted(Path("/content/hanium-lipreading/results").glob("*.json")):
    dst = RESULTS / f.name
    if not dst.exists():
        shutil.copy2(f, dst)
        print("복사", f.name)
print("기준선:", sorted(p.stem for p in RESULTS.glob("*.json")))

In [ ]:
# ═══ 실험 러너 · 화자 독립 교차검증 ═══
# NAME은 매번 바꾼다. 같은 이름에 다른 조건을 쌓으면 비교가 깨지고,
# 같은 조건을 다시 쌓으면 이미 잰 것을 또 재게 된다.
NAME     = "이름을바꿔라"   # 예: no5_192 · lm40 · ep120
BASELINE = "cv192"          # 견줄 기준선. results/<이름>.json 이 있어야 한다
SEEDS    = "42 1 7"         # 한 칸 3시드가 최소 단위. 1시드로는 방향만 본다
EXTRA    = ""               # 예: "--epochs 120" · "--exclude s05" · "--speakers s01"

assert NAME != "이름을바꿔라", "NAME을 이번 실험 이름으로 바꿔라"
assert not (RESULTS / (NAME + ".json")).exists(), (
    NAME + ".json이 이미 있다. 이어받기라면 이 줄을 지우고, 새 실험이면 이름을 바꿔라"
)

cmd = (
    f"python scripts/run_experiment.py --name {NAME} --seeds {SEEDS}"
    f" --baseline {BASELINE} --results-dir {RESULTS}"
    f" --manifest {manifest_f60} --data-root {TRAIN_ROOT_F60}"
    f" --checkpoint-dir {DRIVE_CHECKPOINTS} --wandb-project lipreading {EXTRA}"
)
print(cmd)
!{cmd}

In [ ]:
# 학습 없이 요약만 다시 본다. 기준선과 견주면 화자별 차이·짝 t검정·판정선까지 찍는다.
cmd = (
    f"python scripts/run_experiment.py --name {NAME} --summary"
    f" --baseline {BASELINE} --results-dir {RESULTS} --manifest {manifest_f60}"
)
print(cmd)
!{cmd}

## 4. 체크포인트 확인

In [ ]:
import torch

# 이름을 바꿔서 본다. cv192_s08_seed42.pt 처럼.
CKPT = DRIVE_CHECKPOINTS / "best.pt"
checkpoint = torch.load(CKPT, map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)

## 5. 증강 실험

2026-08-15에 공간 증강은 기각됐다(0.324 대 0.323).

### 증강 패치 원상복구

In [ ]:
import importlib, sys
from src.ml.preprocess.augmentation import pipeline

fresh = importlib.reload(pipeline)                    # 소스에서 원본을 새로 읽음
patched = sys.modules["src.ml.training.train"].VideoAugmentation
patched.__call__ = fresh.VideoAugmentation.__call__   # 학습 코드가 쥔 클래스에 되돌림
print("복구:", patched.__call__.__qualname__)          # VideoAugmentation.__call__ 이면 정상

### 시간축 증강 (s06 · 3시드)

In [ ]:
# ═══ 시간축 증강 실험 · 이 셀 하나만 실행 (재실행 안전) ═══
from pathlib import Path
import numpy as np
from src.ml.preprocess.augmentation.pipeline import VideoAugmentation
from src.ml.training.train import train

DRIVE_ROOT        = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive/hanium-lipreading"))
DRIVE_CHECKPOINTS = globals().get("DRIVE_CHECKPOINTS", DRIVE_ROOT / "checkpoints")
manifest_f60      = globals().get("manifest_f60", DRIVE_ROOT / "manifest_f60.csv")
TRAIN_ROOT_F60    = globals().get("TRAIN_ROOT_F60", Path("/content/data_f60"))
assert manifest_f60.exists(), f"매니페스트 없음: {manifest_f60}"

TIME_CROP_PROB = 0.5
TIME_CROP_MIN  = 0.75
SEEDS = [42, 1, 7]

# 원본을 클래스 속성에 한 번만 보관 → 몇 번 실행해도 진짜 원본이 유지된다
if not getattr(VideoAugmentation, "_taug_patched", False):
    VideoAugmentation._taug_orig = VideoAugmentation.__call__
    VideoAugmentation._taug_patched = True
ORIG = VideoAugmentation._taug_orig
assert ORIG.__qualname__ == "VideoAugmentation.__call__", f"원본이 아님: {ORIG.__qualname__}"

def call_with_time_aug(self, clip, return_details=False):
    if return_details:
        return ORIG(self, clip, True)
    frames = ORIG(self, clip, False)
    if self.rng.random() < TIME_CROP_PROB:
        T = len(frames)
        keep = int(T * self.rng.uniform(TIME_CROP_MIN, 1.0))
        if 2 <= keep < T:
            start = int(self.rng.integers(0, T - keep + 1))
            idx = np.linspace(start, start + keep - 1, T).round().astype(int)
            frames = frames[idx]
    return frames

VideoAugmentation.__call__ = call_with_time_aug
print(f"시간축 증강 적용 · 확률 {TIME_CROP_PROB} · 크롭 하한 {TIME_CROP_MIN}")
print(f"데이터 {TRAIN_ROOT_F60}\n")

results = {}
try:
    for seed in SEEDS:
        print(f"\n{'='*16} s06 · seed {seed} · 시간축 증강 {'='*16}")
        results[seed] = train(
            manifest_f60=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4,
            seed=seed, val_speakers=["s06"],
            checkpoint_path=DRIVE_CHECKPOINTS / f"taug_192_s06_seed{seed}.pt",
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name=f"taug_192_s06_seed{seed}",
        )["best"]
finally:
    VideoAugmentation.__call__ = ORIG
    print("\n[증강 원상복구 완료]")

v = [results[s] for s in SEEDS if s in results]
print(f"\n{'='*56}")
if len(v) == len(SEEDS):
    print(f"시간축 증강   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
else:
    print(f"완료 {len(v)}/{len(SEEDS)}시드   {[f'{x:.3f}' for x in v]}")
# 아래는 30프레임 시절 기준선이다. 60프레임 · 192x96에서는 다시 재야 한다.
print(f"기준선(30프레임 시절)  ['0.732', '0.783', '0.745']  평균 0.754")
print(f"지금 규격의 기준선은 없다. align 3시드가 나오면 그것과 견줄 것.")

### 재현성 확인 — 같은 시드 재실행

In [ ]:
from src.ml.training.train import train
from src.ml.preprocess.augmentation import VideoAugmentation

# 같은 시드를 다시 돌려 값이 얼마나 흔들리는지 잰다.
# s06 seed42가 0.809 / 0.739 / 0.822로 갈렸다. 폭 0.083이다.
#
# 한때 "시간축 증강 패치 오염"으로 봤으나 2026-08-20 2차 정정에서 철회했다.
# 원인은 오염이 아니라 재현 불가 자체이며, 남은 후보가 cuDNN 비결정성이다.
# True로 두 번 돌려 폭이 좁아지는지 본다. 켜면 느려진다.
DETERMINISTIC = False

print("call:", VideoAugmentation.__call__.__qualname__)
assert VideoAugmentation.__call__.__qualname__ == "VideoAugmentation.__call__", "패치가 걸려 있다"

tag = "repro_192_s06_seed42" + ("_det" if DETERMINISTIC else "")
acc = train(
    manifest_f60=manifest_f60, data_root=TRAIN_ROOT_F60,
    epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
    val_speakers=["s06"],
    checkpoint_path=DRIVE_CHECKPOINTS / (tag + ".pt"),
    num_workers=8, amp=True, ema_decay=0.998,
    hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
    deterministic=DETERMINISTIC,
    wandb_project="lipreading", run_name=tag)["best"]

print("")
print("재현 결과 ", round(acc, 3), "· cuDNN 결정성", "켬" if DETERMINISTIC else "끔")
print("  기존 관측  0.809 / 0.739 / 0.822  (폭 0.083)")
print("")
print("  한 번 돌려서는 아무 판정도 못 한다. 같은 설정으로 두 번 이상 돌린 값의")
print("  폭을 비교할 것. 결정성을 켠 쪽 폭이 눈에 띄게 좁으면 cuDNN이 원인이다.")

## 6. 랜드마크 실험

2화자 3시드에서 −0.023. 융합은 효과가 없었고 랜드마크 단독은 0.397이었다.

### 보조 특징(aux_f60.npz) 추출과 모델 패치

In [ ]:
import sys, urllib.request, unicodedata, time
import numpy as np, cv2, torch
from pathlib import Path
from torch import nn

from src.ml.preprocess.lip_crop import create_landmarker, LIP_LANDMARKS, lip_openness
from src.ml.preprocess.normalize import trim_to_speech, resample_frames
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.models.backbone import LipReadingBackbone
from src.ml.models.temporal import TemporalBiGRU
from src.ml.models.classification_head import ClassificationHead
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

AUX_DIM, PROJ_DIM, FRAMES = 44, 128, 60
OUT  = DRIVE_ROOT / "aux_f60.npz"
PART = DRIVE_ROOT / "aux_f60_partial.npz"
EXT  = [".mp4", ".avi", ".mov"]

if not OUT.exists():
    MODEL = Path("/content/hanium-lipreading/models/face_landmarker.task")
    if not MODEL.exists():
        MODEL.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", str(MODEL))
        print("랜드마커 모델 내려받음")

    cand = []
    for p in sorted(DRIVE_ROOT.iterdir()):
        if not p.is_dir(): continue
        for d in [p] + [q for q in sorted(p.iterdir()) if q.is_dir()]:
            v = [q for q in d.iterdir() if q.suffix.lower() in EXT]
            if v: cand.append([len(v), d])
    cand.sort(reverse=True, key=lambda x: x[0])
    assert cand, "원본 영상을 Drive에서 못 찾았다"
    RAW_DIR = cand[0][1]
    print("RAW_DIR", RAW_DIR, cand[0][0], "개")

    vids = dict()
    for p in sorted(RAW_DIR.iterdir()):
        if p.suffix.lower() in EXT:
            vids[unicodedata.normalize("NFC", p.stem)] = p
    stems = [unicodedata.normalize("NFC", q.stem) for q in sorted(PROCESSED_F60.glob("*.npy"))]
    hit = [s for s in stems if s in vids]
    print("npy", len(stems), "개 · 원본 매칭", len(hit), "개")
    assert len(hit) > len(stems) * 0.95, "RAW_DIR 매칭 실패"

    names, feats = [], []
    if PART.exists():
        _p = np.load(PART, allow_pickle=False)
        names, feats = list(_p["names"]), list(_p["feats"])
        print("이어받기", len(names), "개")
    done = set(names)
    lmk = create_landmarker()
    t0 = time.time()
    try:
        for k, stem in enumerate(hit):
            if stem in done: continue
            cap = cv2.VideoCapture(str(vids[stem]))
            pts, ops = [], []
            while True:
                ok, fr = cap.read()
                if not ok: break
                h, w = fr.shape[:2]
                import mediapipe as mp
                res = lmk.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
                if not res.face_landmarks: continue
                L = res.face_landmarks[0]
                pts.append([[L[i].x * w, L[i].y * h] for i in LIP_LANDMARKS])
                ops.append(lip_openness(L, w, h))
            cap.release()
            if len(pts) < 2: continue
            idx = resample_frames(trim_to_speech(list(range(len(pts))), ops), FRAMES)
            P = np.array(pts, dtype=np.float32)[idx]
            O = np.array(ops, dtype=np.float32)[idx]
            P = P - P.mean(axis=1, keepdims=True)
            scale = float(np.median(np.linalg.norm(P[:, 0] - P[:, 10], axis=1))) + 1e-6
            P = P / scale
            dur = np.full((FRAMES, 1), len(pts) / float(FRAMES), dtype=np.float32)
            feats.append(np.concatenate([P.reshape(FRAMES, 42), O.reshape(FRAMES, 1), dur], axis=1).astype(np.float32))
            names.append(stem)
            if len(names) % 100 == 0:
                np.savez(PART, names=np.array(names), feats=np.array(feats))
                print(len(names), "/", len(hit), "·", round(time.time() - t0), "초")
    finally:
        lmk.close()
    np.savez(OUT, names=np.array(names), feats=np.array(feats, dtype=np.float32))
    print("저장", OUT, len(names), "개 ·", round(time.time() - t0), "초")

_z = np.load(OUT, allow_pickle=False)
AUX_FEATS = _z["feats"].astype(np.float32)
AUX_INDEX = dict(zip(list(_z["names"]), range(len(_z["names"]))))
print("보조 특징", AUX_FEATS.shape)

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(LipReadingModel, "_orig_init"):
    LipReadingModel._orig_init = LipReadingModel.__init__
    LipReadingModel._orig_forward = LipReadingModel.forward
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch

MISS = []

def getitem_aux(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    name = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    j = AUX_INDEX.get(name, -1)
    if j < 0:
        MISS.append(name)
        a = np.zeros((frames.shape[1], AUX_DIM), dtype=np.float32)
    else:
        a = AUX_FEATS[j]
    return frames, torch.from_numpy(a), label

def init_aux(self, num_classes, hidden_dim=256, num_layer=2, dropout=0.2,
             pretrained=False, freeze_backbone=False):
    nn.Module.__init__(self)
    self.backbone = LipReadingBackbone(pretrained=pretrained)
    if freeze_backbone:
        self.backbone.freeze_resnet()
    self.aux_proj = nn.Sequential(nn.Linear(AUX_DIM, PROJ_DIM), nn.ReLU())
    self.temporal = TemporalBiGRU(
        input_dim=self.backbone.feature_dim + PROJ_DIM,
        hidden_dim=hidden_dim, num_layer=num_layer, dropout=dropout)
    self.head = ClassificationHead(
        input_dim=self.temporal.output_dim, num_classes=num_classes, dropout=dropout)

def forward_aux(self, frames, aux):
    f = self.backbone(frames)
    f = torch.cat([f, self.aux_proj(aux)], dim=2)
    return self.head(self.temporal(f))

def run_epoch_aux(model, loader, criterion, device, optimizer=None,
                  amp=False, grad_clip=None, averager=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.set_grad_enabled(is_training):
        for frames, aux, labels in loader:
            frames = frames.to(device, non_blocking=True)
            aux = aux.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames, aux)
                loss = criterion(logits, labels)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None:
                    averager.update(model)
            total_loss += loss.float().item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += labels.size(0)
    return total_loss / total_count, total_correct / total_count

LipReadingDataset.__getitem__ = getitem_aux
LipReadingModel.__init__ = init_aux
LipReadingModel.forward = forward_aux
T.run_epoch = run_epoch_aux
print("패치 적용 완료")
# end

### 랜드마크 추가 학습 (s05 · s06 · seed 42)

In [ ]:
res = []
for sp in ["s05", "s06"]:
    print("")
    print("======== 랜드마크 추가 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_f60=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("lm_192_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="lm_192_" + sp)["best"]
    res.append([sp, round(acc, 4)])
    print("누적:", res)
print("")
print("좌표 못 찾은 클립:", len(set(MISS)))
# end


### 랜드마크 추가 학습 (시드 1 · 7)

In [ ]:
res2 = []
for sp in ["s05", "s06"]:
    for sd in [1, 7]:
        print("")
        print("======== 랜드마크 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_f60=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("lm_192_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="lm_192_" + sp + "_seed" + str(sd))["best"]
        res2.append([sp, sd, round(acc, 4)])
        print("누적:", res2)
# end

### 랜드마크 단독 분류 (영상 없이)

In [ ]:
import csv, unicodedata, numpy as np, torch
from torch import nn
from pathlib import Path

Z = np.load(DRIVE_ROOT / "aux_f60.npz", allow_pickle=False)
FEAT = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["feats"].astype(np.float32)))
rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
X, Y, S = [], [], []
for r in rows:
    stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
    if stem not in FEAT: continue
    X.append(FEAT[stem]); Y.append(int(r["label_id"])); S.append(r["speaker_id"])
X = torch.tensor(np.stack(X)); Y = torch.tensor(Y); S = np.array(S)
NC = int(Y.max()) + 1
print("클립", len(X), "· 차원", tuple(X.shape[1:]), "· 클래스", NC)

class AuxOnly(nn.Module):
    def __init__(self, nclass, dim=128):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(44, dim), nn.ReLU())
        self.gru = nn.GRU(dim, dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.head = nn.Linear(dim * 2, nclass)
    def forward(self, x):
        h, _ = self.gru(self.proj(x))
        return self.head(h.mean(dim=1))

# 영상 기준선. 옛 규격(base60) 숫자를 박아 두었던 자리다. cv192에서 읽는다.
import json as _json
from pathlib import Path as _Path
_p = DRIVE_ROOT / "results" / "cv192.json"
if not _p.exists():
    _p = _Path("/content/hanium-lipreading/results/cv192.json")
_r = _json.loads(_p.read_text(encoding="utf-8"))
BASE = {}
for _sp in sorted({x["speaker"] for x in _r}):
    _v = [x["best"] for x in _r if x["speaker"] == _sp]
    BASE[_sp] = round(sum(_v) / len(_v), 3)
print("영상 기준선 cv192:", BASE)
dev = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, BS, SEED = 80, 16, 42
print("")
print("화자   랜드마크단독(저장)  (마지막)   영상기준선   우연")
res = []
for sp in sorted(set(S)):
    tr, va = np.where(S != sp)[0], np.where(S == sp)[0]
    mu = X[tr].reshape(-1, 44).mean(0); sg = X[tr].reshape(-1, 44).std(0) + 1e-6
    xt, yt = ((X[tr] - mu) / sg).to(dev), Y[tr].to(dev)
    xv, yv = ((X[va] - mu) / sg).to(dev), Y[va].to(dev)
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = AuxOnly(NC).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    g = torch.Generator().manual_seed(SEED)
    hist, best = [], 0.0
    for ep in range(EPOCHS):
        m.train()
        for i in torch.randperm(len(xt), generator=g).split(BS):
            opt.zero_grad(set_to_none=True)
            loss = crit(m(xt[i]), yt[i]); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        m.eval()
        with torch.no_grad():
            acc = (m(xv).argmax(1) == yv).float().mean().item()
        hist.append(acc)
        best = max(best, float(np.mean(hist[-3:])))
    res.append([sp, round(best, 4), round(hist[-1], 4)])
    print(" ", sp, "   ", round(best, 3), "        ", round(hist[-1], 3), "     ",
          BASE.get(sp, 0), "     0.067")
print("")
print("평균  저장", round(np.mean([r[1] for r in res]), 3),
      "· 마지막", round(np.mean([r[2] for r in res]), 3),
      "· 영상기준선", round(np.mean(list(BASE.values())), 3))
# end

## 7. 화자 제외 실험

s05를 학습·평가 양쪽에서 뺀다. 7화자 1시드 +0.075.

In [ ]:
from src.ml.training import train as T
# end
import csv
(DRIVE_ROOT / "no5_192_results.json").unlink(missing_ok=True)
# end
rows = list(csv.DictReader(open(manifest_f60, encoding="utf-8")))
keep = [r for r in rows if r["speaker_id"] != "s05"]
manifest_no5 = DRIVE_ROOT / "manifest_f60_no_s05_3.csv"
with open(manifest_no5, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["clip_path", "label_id", "label_text", "speaker_id", "take"])
    w.writeheader()
    for r in keep: w.writerow(r)
print("클립", len(rows), "→", len(keep), "· 문구", len(set(r["label_text"] for r in keep)), "· 화자", len(set(r["speaker_id"] for r in keep)))

LOG5 = DRIVE_ROOT / "no5_192_results.json"
import json
done = json.loads(LOG5.read_text()) if LOG5.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s06", "s07", "s08", "s09"]:
    for sd in [1, 7]:
        if (sp, sd) in seen: continue
        print("")
        print("======== s05 제외 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_f60=manifest_no5, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("no5_192_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="no5_192_" + sp + "_seed" + str(sd))["best"]
        done.append([sp, sd, round(acc, 4)])
        LOG5.write_text(json.dumps(done))
        print("누적:", done)
# end

클립 1238 → 1088 · 문구 15 · 화자 7

======== s05 제외 · s01 · seed 1 ========


lr,██▇▇▆▅▄▂▁
train/acc,▁▁▂▄▅▆▇██
train/loss,█▇▆▅▄▃▂▁▁
val/acc,▅▁▁▁▁▁███
val/acc_smoothed,▅▃▂▁▁▁▃▆█
val/loss,▁▁▁▁▁▂▄▆█
lr,0.0002
train/acc,0.92322
train/loss,1.8363
val/acc,0.07643
val/acc_smoothed,0.07643


장치: cuda | 클래스: 15개
학습 931개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7071 acc 0.059 | val loss 2.7073 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.6296 acc 0.128 | val loss 2.7060 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.5197 acc 0.215 | val loss 2.7072 acc 0.070 avg 0.070 | lr 1.99e-04
[  4/80] train loss 2.3419 acc 0.431 | val loss 2.7124 acc 0.076 avg 0.072 | lr 1.99e-04
[  5/80] train loss 2.1545 acc 0.679 | val loss 2.7220 acc 0.076 avg 0.074 | lr 1.98e-04
[  6/80] train loss 2.0050 acc 0.815 | val loss 2.7387 acc 0.076 avg 0.076 | lr 1.97e-04
[  7/80] train loss 1.8990 acc 0.896 | val loss 2.7639 acc 0.076 avg 0.076 | lr 1.96e-04
[  8/80] train loss 1.8234 acc 0.948 | val loss 2.8028 acc 0.076 avg 0.076 | lr 1.95e-04
[  9/80] train loss 1.7848 acc 0.968 | val loss 

lr,██████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▄▆▇███████████████████████████████████
train/loss,█▇▄▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▂▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇██████████▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▄▅▅▅▆▆▆▆▇▇▇████████▇▇▇▇▇▇▇
val/loss,▆▆▆▆▆▇▇███▆▆▅▅▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.6879
best_val_acc_smoothed,0.68577
lr,0
train/acc,1
train/loss,1.67099


누적: [['s01', 1, 0.6879]]

======== s05 제외 · s01 · seed 7 ========


장치: cuda | 클래스: 15개
학습 931개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7142 acc 0.066 | val loss 2.7100 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.6331 acc 0.133 | val loss 2.7089 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/80] train loss 2.5295 acc 0.224 | val loss 2.7078 acc 0.064 avg 0.064 | lr 1.99e-04
[  4/80] train loss 2.3529 acc 0.412 | val loss 2.7094 acc 0.070 avg 0.066 | lr 1.99e-04
[  5/80] train loss 2.1307 acc 0.697 | val loss 2.7147 acc 0.070 avg 0.068 | lr 1.98e-04
[  6/80] train loss 1.9848 acc 0.834 | val loss 2.7261 acc 0.070 avg 0.070 | lr 1.97e-04
[  7/80] train loss 1.8791 acc 0.902 | val loss 2.7395 acc 0.070 avg 0.070 | lr 1.96e-04
[  8/80] train loss 1.8181 acc 0.951 | val loss 2.7623 acc 0.070 avg 0.070 | lr 1.95e-04


## 8. 오디오 증류 실험

7화자에서 −0.040. 소리의 헷갈림이 입 모양과 맞지 않았다.

In [ ]:
import csv, json, unicodedata, subprocess, numpy as np, torch
from pathlib import Path
from torch import nn
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

TEACH = DRIVE_ROOT / "teacher_f60.npz"
RAW_DIR = DRIVE_ROOT / "raw"
WHISPER = "openai/whisper-small"
TAU, ALPHA = 2.0, 0.5

rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
PH = [t for _, t in sorted(set((int(r["label_id"]), r["label_text"]) for r in rows))]
print("문구", len(PH), "개")

if not TEACH.exists():
    subprocess.run(["pip", "install", "--quiet", "transformers", "accelerate"], check=False)
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from transformers.modeling_outputs import BaseModelOutput
    dev = "cuda"
    proc = WhisperProcessor.from_pretrained(WHISPER)
    proc.tokenizer.set_prefix_tokens(language="korean", task="transcribe")
    wm = WhisperForConditionalGeneration.from_pretrained(WHISPER).to(dev).eval()

    seqs = [proc.tokenizer(p).input_ids for p in PH]
    L = max(len(s) for s in seqs)
    pad = proc.tokenizer.pad_token_id if proc.tokenizer.pad_token_id is not None else proc.tokenizer.eos_token_id
    lab = torch.full((len(PH), L), pad, dtype=torch.long)
    msk = torch.zeros(len(PH), L - 1)
    for i, s in enumerate(seqs):
        lab[i, :len(s)] = torch.tensor(s)
        msk[i, 3:len(s) - 1] = 1.0
    lab, msk = lab.to(dev), msk.to(dev)
    din, tgt = lab[:, :-1], lab[:, 1:]

    names, probs, nofail = [], [], 0
    tmp = Path("/content/_a.wav")
    for k, r in enumerate(rows):
        stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
        src = None
        for e in [".mp4", ".avi", ".mov"]:
            c = RAW_DIR / (stem + e)
            if c.exists(): src = c; break
        if src is None: continue
        subprocess.run(["ffmpeg", "-y", "-loglevel", "quiet", "-i", str(src),
                        "-ac", "1", "-ar", "16000", str(tmp)], check=False)
        if not tmp.exists() or tmp.stat().st_size < 2000:
            nofail += 1; continue
        import soundfile as sf
        wav, sr = sf.read(str(tmp))
        f = proc(wav, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad():
            enc = wm.model.encoder(f).last_hidden_state.repeat(len(PH), 1, 1)
            out = wm(decoder_input_ids=din, encoder_outputs=BaseModelOutput(last_hidden_state=enc))
            lp = torch.log_softmax(out.logits.float(), dim=-1)
            tok = lp.gather(2, tgt.unsqueeze(2)).squeeze(2)
            sc = (tok * msk).sum(1) / msk.sum(1)
            p = torch.softmax(sc / TAU, dim=0).cpu().numpy()
        names.append(stem); probs.append(p)
        tmp.unlink(missing_ok=True)
        if len(names) % 200 == 0: print(len(names), "/", len(rows))
    np.savez(TEACH, names=np.array(names), probs=np.array(probs, dtype=np.float32))
    print("교사 저장", len(names), "개 · 오디오 실패", nofail)

Z = np.load(TEACH, allow_pickle=False)
TP = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["probs"]))
lid = dict(zip([unicodedata.normalize("NFC", Path(r["clip_path"]).stem) for r in rows],
               [int(r["label_id"]) for r in rows]))
top1 = np.mean([1.0 * (int(np.argmax(TP[s])) == lid[s]) for s in TP])
ent = np.mean([-float((TP[s] * np.log(TP[s] + 1e-9)).sum()) for s in TP])
print("")
print("교사 정확도", round(float(top1), 3), "· 평균 엔트로피", round(float(ent), 3),
      "· 균일분포 엔트로피", round(float(np.log(len(PH))), 3))

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch
UNIF = np.full(len(PH), 1.0 / len(PH), dtype=np.float32)

def getitem_kd(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    stem = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    return frames, torch.from_numpy(TP.get(stem, UNIF).copy()), label

def run_epoch_kd(model, loader, criterion, device, optimizer=None,
                 amp=False, grad_clip=None, averager=None):
    training = optimizer is not None
    model.train(training)
    tl = tc = tn = 0
    with torch.set_grad_enabled(training):
        for frames, soft, labels in loader:
            frames = frames.to(device, non_blocking=True)
            soft = soft.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames)
                ce = criterion(logits, labels)
                kd = -(soft * torch.log_softmax(logits.float(), dim=1)).sum(1).mean()
                loss = (1.0 - ALPHA) * ce + ALPHA * kd
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None: averager.update(model)
            tl += loss.float().item() * labels.size(0)
            tc += (logits.argmax(1) == labels).sum().item()
            tn += labels.size(0)
    return tl / tn, tc / tn

LipReadingDataset.__getitem__ = getitem_kd
T.run_epoch = run_epoch_kd
print("증류 패치 적용 · alpha", ALPHA, "· tau", TAU)

LOGK = DRIVE_ROOT / "kd_192_results.json"
done = json.loads(LOGK.read_text()) if LOGK.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]:
    if (sp, 42) in seen: continue
    print("")
    print("======== 증류 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_f60=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("kd_192_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="kd_192_" + sp)["best"]
    done.append([sp, 42, round(acc, 4)])
    LOGK.write_text(json.dumps(done))
    print("누적:", done)
# end

## 부록 · 오답 분석 (손으로 실행)

**"모두 실행"은 아래 정지 셀에서 멈춘다.** 여기까지가 파이프라인이고, 이 아래는
체크포인트가 이미 있을 때 따로 돌리는 분석 도구다.

옛 규격(112x80 · 30프레임) 전처리·매니페스트·복사 셀은 2026-08-26에 지웠다.
현행 절차는 `8-2. 셋업을 한 셀로`가 전부 대신한다. 필요하면 git 이력에서 꺼낼 것.

In [ ]:
# ═══ 여기서 멈춘다 ═══
# 아래 오답 분석 셀은 체크포인트를 읽는다. "모두 실행"에 딸려 돌면
# 없는 파일을 찾다 멈추므로 여기서 의도적으로 끊는다.
raise RuntimeError("의도된 정지. 여기까지가 현재 파이프라인이다. "
                   "아래 분석 셀은 필요할 때 손으로 하나씩 돌릴 것.")

### 어느 문구끼리 헷갈리는가

어느 문구끼리 헷갈리는지 본다.


### 예측 편향과 온도(τ) 보정

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.training.dataset import LipReadingDataset
from src.ml.training.train import split_by_speaker

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
TAU_MAIN = 1.0          # 사전 선택값. 이걸로 판정한다

def probs_fold(speaker):
    ds = LipReadingDataset(manifest_f60, TRAIN_ROOT_F60)
    _, vi, _ = split_by_speaker(ds, val_speakers=[speaker])
    loader = DataLoader(Subset(ds, vi), batch_size=16, num_workers=2)
    ck = torch.load(DRIVE_CHECKPOINTS / f"cv192_{speaker}_seed42.pt", map_location="cuda")
    m = LipReadingModel(num_classes=ck["num_classes"], hidden_dim=ck["hidden_dim"],
                        num_layer=ck["num_layer"], dropout=ck["dropout"]).cuda()
    m.load_state_dict(ck["model_state"]); m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                o = m(x.cuda())
            P.append(F.softmax(o.float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y), sorted({r["label_text"] for r in ds.rows})

fold = {}
for sp in SPEAKERS:
    fold[sp] = probs_fold(sp)
    print(f"{sp} 완료")

texts = fold["s01"][2]
C = len(texts)
total = sum(len(fold[s][1]) for s in SPEAKERS)

# ── 1. 예측 분포가 얼마나 치우쳤나 ──
pred_n, true_n = torch.zeros(C), torch.zeros(C)
for P, Y, _ in fold.values():
    pred_n += torch.bincount(P.argmax(1), minlength=C).float()
    true_n += torch.bincount(Y, minlength=C).float()

print(f"\n{'문구':<16}{'정답':>6}{'예측':>6}{'배율':>7}")
print("-" * 37)
for i in torch.argsort(pred_n / true_n, descending=True):
    print(f"{texts[i]:<16}{int(true_n[i]):>6}{int(pred_n[i]):>6}{pred_n[i] / true_n[i]:>7.2f}")

# ── 2. 보정 ──
def freq(speakers):
    n = torch.zeros(C)
    for s in speakers:
        n += torch.bincount(fold[s][0].argmax(1), minlength=C).float()
    return (n / n.sum()).clamp(min=1e-6)

base = sum((fold[s][0].argmax(1) == fold[s][1]).sum().item() for s in SPEAKERS) / total

print(f"\n{'τ':>5}{'상한(자기 화자)':>16}{'정직(타화자 추정)':>18}")
print("-" * 40)
for tau in (0.0, 0.25, 0.5, 0.75, 1.0):
    hi = ho = 0
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        others = [s for s in SPEAKERS if s != sp]
        hi += ((P / freq([sp]) ** tau).argmax(1) == Y).sum().item()
        ho += ((P / freq(others) ** tau).argmax(1) == Y).sum().item()
    mark = "  ← 판정" if tau == TAU_MAIN else ""
    print(f"{tau:>5.2f}{hi / total:>16.3f}{ho / total:>18.3f}{mark}")

print(f"\n보정 없음  {base:.3f}")

# ── 3. τ=1 정직 버전, 화자별 ──
print(f"\n{'화자':<6}{'보정전':>8}{'보정후':>8}{'차이':>8}")
print("-" * 32)
for sp in SPEAKERS:
    P, Y, _ = fold[sp]
    others = [s for s in SPEAKERS if s != sp]
    b = (P.argmax(1) == Y).float().mean().item()
    a = ((P / freq(others) ** TAU_MAIN).argmax(1) == Y).float().mean().item()
    print(f"{sp:<6}{b:>8.3f}{a:>8.3f}{a - b:>+8.3f}")

### 문구별 재현율과 주요 오답

In [ ]:
import collections

for target_name in ["도와주세요", "자세바꿔주세요", "숨쉬기힘들어요"]:
    t = texts.index(target_name)
    print(f"\n=== {target_name} ===")
    print(f"{'화자':<6}{'정답':>6}{'맞춤':>6}{'재현율':>8}  주요 오답")
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        mask = Y == t
        n = int(mask.sum())
        if n == 0:
            continue
        pred = P[mask].argmax(1)
        hit = int((pred == t).sum())
        wrong = collections.Counter(texts[int(i)] for i in pred if int(i) != t)
        top = " · ".join(f"{a}×{c}" for a, c in wrong.most_common(2))
        print(f"{sp:<6}{n:>6}{hit:>6}{hit / n:>8.2f}  {top}")


### 문구별 움직임과 클립 중복률

In [ ]:
import numpy as np, csv, collections
from pathlib import Path

with open(manifest_f60, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

acc = collections.defaultdict(list)
for r in rows:
    a = np.load(Path(TRAIN_ROOT_F60) / r["clip_path"])[:, :, :, 0].astype(np.float32)
    motion = np.abs(np.diff(a, axis=0)).mean()
    dup = np.mean([np.array_equal(a[i], a[i + 1]) for i in range(len(a) - 1)])
    acc[r["label_text"]].append((motion, dup))

print(f"{'문구':<16}{'움직임':>8}{'중복률':>8}{'클립':>6}")
print("-" * 40)
for m, ph, d, n in sorted((np.mean([x[0] for x in v]), ph,
                           np.mean([x[1] for x in v]), len(v))
                          for ph, v in acc.items()):
    print(f"{ph:<16}{m:>8.2f}{d:>8.2f}{n:>6}")